# 02 — Skeleton Extraction: FineBadminton

Batch-process FineBadminton frames through YOLOv8-Pose to extract dual-player skeletons.

**Player ordering convention (important):**
- Pose extractor sorts by **Y position** (image coords — Y increases downward)
- **Player 0** (nodes 0–16)  = top-court player (smaller Y centroid)
- **Player 1** (nodes 17–33) = bottom-court player (larger Y centroid)
- At dataset-load time (`dataset.py`), the hitter is always moved to player-0 position
  using the `"hitter"` field ("top"/"bottom") from FineBadminton annotations.

**Steps:**
1. (Optional) Clean old skeletons
2. Process all FineBadminton frames → per-rally skeleton .npy files (Y-sorted)
3. Verify Y-sort ordering and hitter-first reordering
4. Compute enriched node features (L0–L3) and visualise

In [ ]:
!pip install tqdm

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import cv2
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm
import matplotlib.pyplot as plt

from src.config import *
from src.config import get_config
from src.data.pose_extractor import PoseExtractor
from src.data.graph_builder import GraphBuilder
from src.data.feature_eng import FeatureEngineer

cfg = get_config()


## 0. (Re-)Extraction Cleanup

Run this cell only if skeletons were previously saved with **X-based** player sorting
(the old default). It deletes existing `.npy` files so the extraction cells below
re-run with the corrected **Y-based** sort.

In [ ]:
FORCE_REEXTRACT = False  # ← set True to re-extract (deletes existing .npy files)

if FORCE_REEXTRACT:
    fb_deleted = list(FB_SKELETONS.glob('*.npy'))
    for f in fb_deleted:
        f.unlink()
    print(f"Deleted {len(fb_deleted)} FB skeletons")
else:
    print("Skipping cleanup — existing skeletons retained.")


## MVP Mode

Set `MVP_MODE = True` to extract just a few rallies/matches and verify the full
pipeline before committing to the full ~2.5 h extraction.

In [ ]:
MVP_MODE    = False  # ← True to test with a few rallies first
MVP_RALLIES = 3      # rallies to extract in MVP mode

if MVP_MODE:
    print(f"MVP mode ON — extracting {MVP_RALLIES} rallies")
else:
    print("Full extraction mode — processing all rallies")


## 1. Extract FineBadminton Skeletons

Frames are pre-provided as `{match}_{rally}_{frame}.jpg`.
Group by rally, extract skeletons, save one `.npy` per rally.

In [ ]:
FB_SKELETONS.mkdir(parents=True, exist_ok=True)

frame_files = sorted(FB_FRAMES.glob('*.jpg'))
print(f'Total FineBadminton frames: {len(frame_files)}')

# Group by rally (first two parts of filename)
rally_frames = defaultdict(list)
for f in frame_files:
    parts = f.stem.split('_')
    if len(parts) >= 3:
        rally_id = f"{parts[0]}_{parts[1]}"  # e.g., '0011_002'
        frame_num = int(parts[2])
        rally_frames[rally_id].append((frame_num, f))

print(f'Rallies found: {len(rally_frames)}')
for rally_id in sorted(rally_frames.keys())[:10]:
    print(f'  {rally_id}: {len(rally_frames[rally_id])} frames')


In [ ]:
# Use yolov8s-pose (24M params) — ~10x faster than yolov8x on CPU.
# Switch to yolov8x-pose for highest accuracy on a GPU.
class _PoseCfg:
    model = 'yolov8s-pose'
    kalman_smoothing = True
    batch_size = 32
    confidence_threshold = 0.3

extractor = PoseExtractor(config=_PoseCfg())

rally_items = sorted(rally_frames.items())
if MVP_MODE:
    rally_items = rally_items[:MVP_RALLIES]
    print(f'MVP: processing {len(rally_items)} rallies → '
          f'{[r for r, _ in rally_items]}')
else:
    already_done = {p.stem for p in FB_SKELETONS.glob('*.npy')}
    rally_items = [(r, v) for r, v in rally_items if r not in already_done]
    print(f'Full mode: {len(rally_items)} rallies to extract '
          f'({len(already_done)} already done)')

for rally_id, frame_list in tqdm(rally_items, desc='FB Rallies'):
    output_path = FB_SKELETONS / f'{rally_id}.npy'
    if output_path.exists():
        print(f'  [SKIP] {rally_id} already extracted')
        continue

    frame_list.sort(key=lambda x: x[0])
    frames = [cv2.imread(str(f)) for _, f in frame_list]

    skeleton = extractor.extract_sequence(frames)
    np.save(output_path, skeleton)
    print(f'  Saved {rally_id}: {skeleton.shape}')

fb_skel_files = sorted(FB_SKELETONS.glob('*.npy'))
print(f'\nSkeleton files so far: {len(fb_skel_files)}')


It took 4 mins for 1 rally 
- ( 1/3 [04:44<09:28, 284.24s/it])
- FB Rallies:  67%|██████▋   | 2/3 [09:01<04:30, 270.50s/it] 

The config uses yolov8x-pose — the extra-large model. You can override it in the notebook without touching config.py:

Option to replace  
extractor = PoseExtractor()  
with 
from src.config import PoseConfig
extractor = PoseExtractor(config=PoseConfig(model="yolov8n-pose"))  # ~10x faster on CPU

- yolov8x-pose (90M params)
- yolov8s-pose (24M params)
- yolov8n-pose (3M params) 

- Old extract_sequence - 1 call per frame	
- New extract_sequence - 1 call per 32 frame batch 


For local MVP testing, yolov8n-pose is the practical choice. For the final full extraction you'd want a GPU (Colab T4), where yolov8x-pose with batching drops to ~10ms/frame.

## 3. Verify Y-Sort and Hitter-First Reordering

Checks both FineBadminton and ShuttleSet skeletons:
- **Y-sort**: player 0 should always have a smaller mean Y than player 1
- **Hitter-first**: after `FineBadmintonDataset` loads a shot, the hitting player should be at nodes 0–16

In [ ]:
from src.data.dataset import FineBadmintonDataset

# ── FineBadminton Y-sort check ───────────────────────────────────────────────
fb_skel_files = sorted(FB_SKELETONS.glob('*.npy'))
print(f"FineBadminton skeletons: {len(fb_skel_files)}")
print("Y-sort check (player 0 should be SMALLER Y than player 1):")
for f in fb_skel_files:
    skel = np.load(f)           # (2, T, 34) — index 1 is Y channel
    p0_y = skel[1, :, :17].mean()
    p1_y = skel[1, :, 17:].mean()
    status = "OK" if p0_y < p1_y else "FAIL"
    print(f"  [{status}] {f.stem}: p0_Y={p0_y:.1f}, p1_Y={p1_y:.1f}")

# ── ShuttleSet Y-sort check ──────────────────────────────────────────────────
ss_skel_files = sorted(SS_SKELETONS.glob('*.npy'))
print(f"\nShuttleSet skeletons: {len(ss_skel_files)}")
if ss_skel_files:
    print("Y-sort check:")
    for f in ss_skel_files:
        skel = np.load(f)
        p0_y = skel[1, :, :17].mean()
        p1_y = skel[1, :, 17:].mean()
        status = "OK" if p0_y < p1_y else "FAIL"
        print(f"  [{status}] {f.stem[:50]}: p0_Y={p0_y:.1f}, p1_Y={p1_y:.1f}")
else:
    print("  (no ShuttleSet skeletons yet — run Section 2 first)")

# ── Hitter-first reordering check (FineBadminton only — needs annotations) ───
print("\nHitter-first check via FineBadmintonDataset (L0 = raw x,y for easy reading):")
fb_ds = FineBadmintonDataset(feature_layer='L0')
real_idx = [i for i, info in enumerate(fb_ds.rally_info) if info['has_skeleton']]
print(f"Samples with real skeletons: {len(real_idx)}")
for i in real_idx[:5]:
    x, y = fb_ds[i]
    info = fb_ds.rally_info[i]
    hitter = info['hitter']
    p0_y = x[1, :, :17].mean().item()
    p1_y = x[1, :, 17:].mean().item()
    # After swap: hitter at p0.
    # "top" hitter → small Y stays at p0  → p0_Y < p1_Y
    # "bottom" hitter → large Y swapped to p0 → p0_Y > p1_Y
    correct = (hitter == 'top' and p0_y < p1_y) or (hitter == 'bottom' and p0_y > p1_y)
    status = "OK" if correct else "FAIL"
    print(f"  [{status}] sample {i}: hitter='{hitter}', "
          f"p0_Y={p0_y:.1f}, p1_Y={p1_y:.1f}, label={IDX_TO_STRATEGY[y]}")

In [ ]:
# ── Skeleton sanity: player movement paths (center-of-mass) ──────────────────
# Each player's CoM = mean of their 17 joints. Plotting this as an x-y path
# on an image-coordinate grid tells you at a glance whether the two players
# are on opposite halves of the court and move sensibly over the rally.

fb_skel_files = sorted(FB_SKELETONS.glob('*.npy'))
if not fb_skel_files:
    print("No skeletons yet — run extraction cells first.")
else:
    skel = np.load(fb_skel_files[0])   # (2, T, 34)  row0=X, row1=Y
    rally_id = fb_skel_files[0].stem
    T = skel.shape[1]

    # Player 0 = top-court (smaller Y);  Player 1 = bottom-court (larger Y)
    p0_x = skel[0, :, :17].mean(axis=1)   # (T,)
    p0_y = skel[1, :, :17].mean(axis=1)
    p1_x = skel[0, :, 17:].mean(axis=1)
    p1_y = skel[1, :, 17:].mean(axis=1)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # ── Left: x-y movement path overlaid on a blank court ────────────────
    ax = axes[0]
    ax.plot(p0_x, p0_y, color='cyan',    lw=1, alpha=0.7, label='P0 – top court')
    ax.plot(p1_x, p1_y, color='magenta', lw=1, alpha=0.7, label='P1 – bottom court')
    # Mark start / end
    ax.scatter(p0_x[0],  p0_y[0],  c='cyan',    s=60, marker='o', zorder=5)
    ax.scatter(p0_x[-1], p0_y[-1], c='cyan',    s=60, marker='X', zorder=5)
    ax.scatter(p1_x[0],  p1_y[0],  c='magenta', s=60, marker='o', zorder=5)
    ax.scatter(p1_x[-1], p1_y[-1], c='magenta', s=60, marker='X', zorder=5)
    ax.invert_yaxis()   # image coords: y increases downward
    ax.set_xlabel('x (pixels)')
    ax.set_ylabel('y (pixels)  ↓')
    ax.set_title(f'Movement paths — {rally_id}\n(circle=start, X=end)')
    ax.legend(fontsize=9)
    ax.set_aspect('equal')
    ax.set_facecolor('#1a1a2e')

    # ── Middle: x position over time ─────────────────────────────────────
    frames = np.arange(T)
    axes[1].plot(frames, p0_x, color='cyan',    lw=1.5, label='P0 x (top)')
    axes[1].plot(frames, p1_x, color='magenta', lw=1.5, label='P1 x (bottom)')
    axes[1].set_xlabel('Frame')
    axes[1].set_ylabel('x (pixels)')
    axes[1].set_title('Lateral position over time')
    axes[1].legend(fontsize=9)

    # ── Right: y position over time (depth — smaller Y = nearer net for top player) ──
    axes[2].plot(frames, p0_y, color='cyan',    lw=1.5, label='P0 y (top court)')
    axes[2].plot(frames, p1_y, color='magenta', lw=1.5, label='P1 y (bottom court)')
    axes[2].invert_yaxis()
    axes[2].set_xlabel('Frame')
    axes[2].set_ylabel('y (pixels)  ↓')
    axes[2].set_title('Depth position over time')
    axes[2].legend(fontsize=9)

    plt.suptitle(
        'Center-of-mass trajectories  '
        '(mean of 17 joints per player — much cleaner than plotting all 17 lines)',
        fontsize=10
    )
    plt.tight_layout()
    plt.show()

    print(f"P0 (top court)    mean Y = {p0_y.mean():.1f}  ← should be SMALLER")
    print(f"P1 (bottom court) mean Y = {p1_y.mean():.1f}  ← should be LARGER")


In [ ]:
# ── Skeleton overlay on actual frames ────────────────────────────────────────
#
# WHAT IS SHOWN
# ─────────────
# Cyan    = Player 0  ← top-court player (smaller Y in image coords)
# Magenta = Player 1  ← bottom-court player (larger Y in image coords)
#
# WHERE TOP/BOTTOM COMES FROM
# ───────────────────────────
# Purely from the extracted image coordinates — pose_extractor.py sorts the two
# detected people by their mean keypoint Y centroid each frame. NO annotations
# are used during extraction. The .npy on disk is always Y-sorted.
#
# WHERE HITTER COMES FROM (FineBadminton only)
# ────────────────────────────────────────────
# The FineBadminton annotations have a "hitter" field ("top"/"bottom") per shot.
# This tells us which court side is hitting, which maps directly to the Y-sorted
# player: "top" hitter → cyan player is hitting; "bottom" hitter → magenta is hitting.
# We highlight the hitting player with a filled bounding box in the shot frames.
# NOTE: the .npy is NOT reordered here — that only happens inside dataset.py at
# load time. Here we read the raw on-disk skeleton and label accordingly.

COCO_EDGES = [
    (0,1),(0,2),(1,3),(2,4),
    (5,6),(5,7),(7,9),(6,8),(8,10),
    (5,11),(6,12),(11,12),
    (11,13),(13,15),(12,14),(14,16),
]
# cyan / magenta (BGR for OpenCV)
P_KP  = [(0,255,255), (255,0,255)]
P_LMB = [(0,150,150), (150,0,150)]
P_LBL = ['P0 top-court', 'P1 bottom-court']

def _draw_player(img, skel_xy, p, is_hitter=False, show_joints=False):
    """Draw one player on img (in-place). skel_xy: (2,34) row0=X row1=Y"""
    off = p * 17
    xs, ys = skel_xy[0, off:off+17], skel_xy[1, off:off+17]

    # Bounding box highlight for the hitter
    if is_hitter:
        valid = [(int(xs[k]), int(ys[k])) for k in range(17) if xs[k]>0 or ys[k]>0]
        if valid:
            x_vals = [pt[0] for pt in valid]; y_vals = [pt[1] for pt in valid]
            cv2.rectangle(img, (min(x_vals)-8, min(y_vals)-8),
                          (max(x_vals)+8, max(y_vals)+8), P_KP[p], 2)

    for i, j in COCO_EDGES:
        x1,y1,x2,y2 = int(xs[i]),int(ys[i]),int(xs[j]),int(ys[j])
        if x1>0 and y1>0 and x2>0 and y2>0:
            cv2.line(img,(x1,y1),(x2,y2),P_LMB[p],2)
    for k in range(17):
        x,y = int(xs[k]),int(ys[k])
        if x>0 or y>0:
            cv2.circle(img,(x,y),4,P_KP[p],-1)
            if show_joints:
                cv2.putText(img,str(k),(x+4,y-4),
                            cv2.FONT_HERSHEY_SIMPLEX,0.32,P_KP[p],1)

    # Player label near hip
    hx = int((xs[11]+xs[12])/2) if xs[11]>0 and xs[12]>0 else int(xs[xs>0].mean() if (xs>0).any() else 0)
    hy = int((ys[11]+ys[12])/2) if ys[11]>0 and ys[12]>0 else int(ys[ys>0].mean() if (ys>0).any() else 0)
    tag = f"{'★ HITTER ' if is_hitter else ''}{P_LBL[p]}"
    cv2.putText(img, tag, (hx-30, hy-14), cv2.FONT_HERSHEY_SIMPLEX, 0.5, P_KP[p], 2)


# ── Build annotation hitter-lookup for FineBadminton ─────────────────────────
# Maps (rally_id, absolute_frame_num) → hitter side ("top" or "bottom")
hitter_lookup = {}   # rally_id → list of (start, end, hitter_side)
if FB_ANNOTATIONS.exists():
    with open(FB_ANNOTATIONS) as f:
        fb_ann = json.load(f)
    for rally in fb_ann:
        rid = rally.get("video","").replace(".mp4","")
        rally_start = rally.get("start_frame", 0)
        windows = []
        for hit in rally.get("hitting", []):
            sf = hit.get("start_frame"); ef = hit.get("end_frame")
            side = hit.get("hitter","")
            if sf is not None and ef is not None and side:
                windows.append((sf, ef, side))
        hitter_lookup[rid] = (rally_start, windows)

def get_hitter_at_frame(rally_id, frame_num):
    """Return 'top', 'bottom', or None for an absolute frame number."""
    if rally_id not in hitter_lookup:
        return None
    _, windows = hitter_lookup[rally_id]
    for sf, ef, side in windows:
        if sf <= frame_num <= ef:
            return side
    return None


# ── Main overlay loop ─────────────────────────────────────────────────────────
fb_skel_files = sorted(FB_SKELETONS.glob('*.npy'))
if not fb_skel_files:
    print("No skeletons yet — run extraction cells first.")
else:
    for skel_file in fb_skel_files:
        rally_id = skel_file.stem
        skel = np.load(skel_file)   # (2, T, 34) — Y-sorted, NOT hitter-reordered
        T = skel.shape[1]

        # Rebuild ordered frame list (frame_num, filepath)
        this_frames = sorted(
            [(int(fp.stem.split('_')[2]), fp)
             for fp in FB_FRAMES.glob('*.jpg')
             if fp.stem.startswith(rally_id + '_')],
            key=lambda x: x[0]
        )
        n = len(this_frames)
        if n == 0:
            print(f"  {rally_id}: no source JPGs — skipping"); continue

        rally_start = hitter_lookup.get(rally_id, (0, []))[0]
        print(f"Rally {rally_id}: T={T}, frames={n}, "
              f"abs_start={rally_start}, annotation windows="
              f"{len(hitter_lookup.get(rally_id,(0,[]))[1])}")

        # ── 8-frame grid ──────────────────────────────────────────────────
        N_SHOW = 8
        idxs = np.linspace(0, min(T,n)-1, N_SHOW, dtype=int)
        fig, axes = plt.subplots(2, 4, figsize=(22, 10))

        for ax, t in zip(axes.flat, idxs):
            abs_frame, fp = this_frames[t]
            img = cv2.imread(str(fp))
            if img is None:
                ax.set_title(f"t={t} missing"); ax.axis('off'); continue

            hitter_side = get_hitter_at_frame(rally_id, abs_frame)
            # "top" hitter → P0 is hitting;  "bottom" hitter → P1 is hitting
            for p in range(2):
                is_hitter = (hitter_side == 'top' and p == 0) or \
                            (hitter_side == 'bottom' and p == 1)
                _draw_player(img, skel[:, t, :], p, is_hitter=is_hitter)

            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            hitter_tag = f" [HITTER={hitter_side}]" if hitter_side else ""
            ax.set_title(f"t={t}{hitter_tag}", fontsize=8)
            ax.axis('off')

        fig.suptitle(
            f"{rally_id} — skeleton overlay\n"
            "Cyan=P0 top-court (nodes 0-16)  |  Magenta=P1 bottom-court (nodes 17-33)\n"
            "★ box = hitter (from annotations). "
            "Top/bottom = image Y coords from YOLOv8, NOT annotations.",
            fontsize=10
        )
        plt.tight_layout()
        plt.show()

        # ── Mid-rally detail frame with joint indices ─────────────────────
        t_mid = min(T,n) // 2
        abs_frame, fp = this_frames[t_mid]
        img = cv2.imread(str(fp))
        hitter_side = get_hitter_at_frame(rally_id, abs_frame)
        for p in range(2):
            is_hitter = (hitter_side == 'top' and p==0) or (hitter_side=='bottom' and p==1)
            _draw_player(img, skel[:, t_mid, :], p, is_hitter=is_hitter, show_joints=True)

        plt.figure(figsize=(14,8))
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.title(
            f"{rally_id} t={t_mid}  —  COCO joint ids + hitter box\n"
            "Cyan=P0 top  |  Magenta=P1 bottom  |  ★ box = hitter from annotation",
            fontsize=10
        )
        plt.axis('off'); plt.tight_layout(); plt.show()


## 4. Feature Engineering Layers

Test all four feature layers (L0–L3) on a real FineBadminton skeleton.

In [ ]:
# Load a sample skeleton and test all feature layers
sample_files = sorted(FB_SKELETONS.glob('*.npy'))

if sample_files:
    skel = np.load(sample_files[0])  # (2, T, 34)
    print(f"Raw skeleton: {skel.shape}")
    
    for layer in ['L0', 'L1', 'L2', 'L3']:
        eng = FeatureEngineer(feature_layer=layer)
        features = eng.compute(skel)
        print(f"  {layer}: {features.shape} — {FEATURE_DIMS[layer]} features/node")
else:
    print("No skeleton files yet. Run extraction cells above.")
    # Test with synthetic data
    skel = np.random.randn(2, 16, 34).astype(np.float32) * 100
    for layer in ['L0', 'L1', 'L2', 'L3']:
        eng = FeatureEngineer(feature_layer=layer)
        features = eng.compute(skel)
        print(f"  {layer}: {features.shape} (synthetic data)")

## 5. Visualise Skeleton Quality

In [ ]:
# Visualize L1 (velocity) features for a sample
if sample_files:
    eng = FeatureEngineer(feature_layer='L1')
    features = eng.compute(skel)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Velocity magnitude per joint over time
    vx = features[2]  # (T, V)
    vy = features[3]
    vel_mag = np.sqrt(vx**2 + vy**2)
    
    # Player 1 wrist velocity (joint 9 = left wrist, 10 = right wrist)
    axes[0].plot(vel_mag[:, 9], label='Left wrist', linewidth=2)
    axes[0].plot(vel_mag[:, 10], label='Right wrist', linewidth=2)
    axes[0].set_title('Player 1 — Wrist Velocity Magnitude')
    axes[0].set_xlabel('Frame')
    axes[0].legend()
    
    # Player 2 wrist velocity
    axes[1].plot(vel_mag[:, 9+17], label='Left wrist', linewidth=2)
    axes[1].plot(vel_mag[:, 10+17], label='Right wrist', linewidth=2)
    axes[1].set_title('Player 2 — Wrist Velocity Magnitude')
    axes[1].set_xlabel('Frame')
    axes[1].legend()
    
    # Velocity heatmap (all joints)
    im = axes[2].imshow(vel_mag.T, aspect='auto', cmap='hot')
    axes[2].set_xlabel('Frame')
    axes[2].set_ylabel('Joint index')
    axes[2].set_title('Velocity Magnitude Heatmap (all joints)')
    axes[2].axhline(y=16.5, color='cyan', linestyle='--', alpha=0.5)
    plt.colorbar(im, ax=axes[2])
    
    plt.tight_layout()
    plt.show()

---
## End-to-End Pipeline Check

Verify the full chain: skeleton → feature engineering → model forward pass.
Run this after the MVP extraction to confirm everything connects correctly.

In [ ]:
import torch
from src.config import get_config
from src.data.dataset import FineBadmintonDataset
from src.data.graph_builder import GraphBuilder
from src.models.stgcn_model import STGCN

cfg = get_config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("=" * 50)
print("End-to-End Pipeline Check")
print("=" * 50)

# 1. Dataset loading with hitter-first reordering
fb_ds = FineBadmintonDataset(feature_layer='L2')
fb_skel_count = sum(1 for s in fb_ds.rally_info if s['has_skeleton'])
print(f"\n1. FineBadminton dataset: {len(fb_ds)} samples, {fb_skel_count} with skeletons")

# 2. Sample a real item and check hitter reordering
real_indices = [i for i, info in enumerate(fb_ds.rally_info) if info['has_skeleton']]
if real_indices:
    idx = real_indices[0]
    x, y = fb_ds[idx]
    info = fb_ds.rally_info[idx]
    print(f"\n2. Sample [{idx}]: rally={info['rally_id']}, hitter='{info['hitter']}', "
          f"label={IDX_TO_STRATEGY[y]}")
    print(f"   Tensor shape: {x.shape}  "
          f"(expected: [{FEATURE_DIMS['L2']}, {cfg.data.shot_window}, 34])")
    print(f"   Non-zero: {(x != 0).sum().item()} / {x.numel()} elements")
    p0_y = x[1, :, :17].mean().item()
    p1_y = x[1, :, 17:].mean().item()
    hitter = info['hitter']
    print(f"   Hitter-first check — p0_Y={p0_y:.1f}, p1_Y={p1_y:.1f}  "
          f"hitter='{hitter}'  {'OK' if hitter else 'hitter field missing'}")
else:
    print("\n2. No real skeletons yet — using synthetic data for shape check")
    x = torch.randn(FEATURE_DIMS['L2'], cfg.data.shot_window, 34)

# 3. Model forward pass
gb = GraphBuilder()
adj = gb.build_adjacency().to(device)
enc = STGCN(
    in_channels=FEATURE_DIMS['L2'],
    num_nodes=NUM_NODES,
    adjacency=adj,
    embedding_dim=256,
).to(device)

with torch.no_grad():
    emb = enc(x.unsqueeze(0).to(device))
print(f"\n3. Model forward pass: {tuple(x.shape)} → embedding {tuple(emb.shape)}")
print("\n✓ All checks passed — ready for SSL pre-training (notebook 03)")

## 6. Test Dataset Loading

In [ ]:
from src.data.dataset import FineBadmintonDataset

fb_ds = FineBadmintonDataset(feature_layer='L2')
print(f"FineBadminton samples: {len(fb_ds)}")

if len(fb_ds) > 0:
    x, y = fb_ds[0]
    print(f"  Sample shape: {x.shape}, label: {y} ({IDX_TO_STRATEGY[y]})")
    print(f"  Has real data: {x.abs().sum() > 0}")


In [ ]:
# Test fold splits
splits = fb_ds.get_fold_splits(n_folds=5)
print(f"Folds: {len(splits)}")
for i, (train, val, test) in enumerate(splits):
    print(f"  Fold {i+1}: train={len(train)}, val={len(val)}, test={len(test)}")

---
## 7. GDINO-Guided Extraction (Contamination Validation)

**Problem:** YOLOv8 "top-2 by confidence" sometimes picks the chair umpire
(seated at x≈102px on the left edge) as P0, displacing the actual top-court player.
Audit shows **14.1% of shot windows** (50/355) have at least one contaminated frame;
`to_create_depth` is worst at 20%.

**Fix:** Run Grounding DINO (`grounding-dino-tiny`) with prompt `"person ."` to
get bounding-box priors for the actual players, then only keep YOLO detections with
IoU ≥ 0.25 against a GDINO prior. Results saved to `finebadminton_skeletons_gdino/`.

**Steps in this section:**
1. Contamination scan — rank all 40 rallies by contamination rate
2. GDINO setup — load model + define extraction helpers
3. GDINO extraction — process `GDINO_RALLIES` and save to `_gdino/` directory
4. Hitter validation comparison — Original vs GDINO per-shot agreement rate

In [ ]:

# ── 7.1  Contamination scan ──────────────────────────────────────────────────
# Flag frames where P0 centroid x < UMPIRE_X_THRESH → likely chair umpire
UMPIRE_X_THRESH = 200   # pixels; umpire sits at x≈102 on 1280px frame

with open(FB_ANNOTATIONS) as f:
    _ann_data = json.load(f)

contamination = []   # (rally_id, total_frames, bad_frames, shot_windows, contaminated_shots)

for rally_data in _ann_data:
    rid  = rally_data.get("video", "").replace(".mp4", "")
    sp   = FB_SKELETONS / f"{rid}.npy"
    if not sp.exists():
        continue

    skel = np.load(sp)                           # (2, T, 34)
    p0_x = skel[0, :, :17].mean(axis=1)         # mean X per frame for P0
    T    = skel.shape[1]
    n_bad_frames = int((p0_x < UMPIRE_X_THRESH).sum())

    rs = rally_data.get("start_frame", 0)
    n_cont_shots, n_shots = 0, 0
    for hit in rally_data.get("hitting", []):
        strats = hit.get("strategies") or []
        if not strats:
            continue
        n_shots += 1
        hf  = hit.get("hit_frame", 0)
        rel = hf - rs
        s   = max(0, rel - 8)
        e   = min(T, s + 16)
        if (p0_x[s:e] < UMPIRE_X_THRESH).any():
            n_cont_shots += 1

    contamination.append((rid, T, n_bad_frames, n_shots, n_cont_shots))

# Sort by contamination rate (shots)
contamination.sort(key=lambda x: x[4] / max(x[3], 1), reverse=True)

print(f"{'Rally':<12} {'Frames':>7} {'Bad%':>6} {'Shots':>6} {'ContShots':>10} {'Shot%':>7}")
print("-" * 55)
for rid, T, nb, ns, nc in contamination:
    bad_pct  = 100 * nb / max(T, 1)
    shot_pct = 100 * nc / max(ns, 1)
    flag = " ⚠" if nc > 0 else ""
    print(f"{rid:<12} {T:>7} {bad_pct:>5.1f}% {ns:>6} {nc:>10} {shot_pct:>6.0f}%{flag}")

# Tallies
total_shots = sum(x[3] for x in contamination)
total_cont  = sum(x[4] for x in contamination)
print(f"\nTotal: {total_cont}/{total_shots} shots contaminated "
      f"({100*total_cont/max(total_shots,1):.1f}%)")
print("\nGDINO_RALLIES (copy-paste into next cell) — top 5 contaminated:")
top5 = [x[0] for x in contamination if x[4] > 0][:5]
print(repr(top5))


In [ ]:

# ── 7.2  Load Grounding DINO ─────────────────────────────────────────────────
# pip install transformers>=5.0.0 timm is included in step 1 (top of notebook)

import torch as _torch
try:
    from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
    import transformers as _tfm
    print(f"transformers {_tfm.__version__}")
except ImportError:
    import subprocess, sys as _sys
    subprocess.check_call([_sys.executable, '-m', 'pip', 'install', '-q',
                           'transformers>=5.0.0', 'timm'])
    from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

from PIL import Image as _PIL_Image

_DEVICE = _torch.device('cuda' if _torch.cuda.is_available() else 'cpu')
_MODEL_ID   = 'IDEA-Research/grounding-dino-tiny'
_BOX_THRESH = 0.30
_TXT_THRESH = 0.25
_GDINO_PROMPT = 'badminton player .'  # more specific than 'person .' — reduces false positives (umpire, ball kids)

_gdino_proc  = AutoProcessor.from_pretrained(_MODEL_ID)
_gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(_MODEL_ID).to(_DEVICE)
_gdino_model.eval()
print(f"Loaded {_MODEL_ID.split('/')[-1]} on {_DEVICE}")

# ── GDINO helpers ─────────────────────────────────────────────────────────────
@_torch.no_grad()
def _gdino_detect(bgr):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    H, W = bgr.shape[:2]
    inp  = _gdino_proc(images=_PIL_Image.fromarray(rgb), text=_GDINO_PROMPT,
                       return_tensors='pt').to(_DEVICE)
    out  = _gdino_model(**inp)
    res  = _gdino_proc.post_process_grounded_object_detection(
        outputs=out,
        input_ids=inp['input_ids'],
        threshold=_BOX_THRESH,
        text_threshold=_TXT_THRESH,
        target_sizes=[(H, W)]
    )[0]
    return res['boxes'].cpu().numpy(), res['scores'].cpu().numpy(), res['labels']

def _iou(b1, b2):
    ix1, iy1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    ix2, iy2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    iw, ih   = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter    = iw * ih
    return inter / ((b1[2]-b1[0])*(b1[3]-b1[1]) + (b2[2]-b2[0])*(b2[3]-b2[1]) - inter + 1e-9)

_yolo_gdino = None
def _get_yolo_gdino():
    global _yolo_gdino
    if _yolo_gdino is None:
        from ultralytics import YOLO
        _yolo_gdino = YOLO('yolov8s-pose.pt')   # x for best accuracy
        print("YOLOv8s-pose loaded")
    return _yolo_gdino

def _extract_gdino_frame(bgr, iou_thresh=0.25):
    """
    Returns (2, 34) skeleton for one frame.
    Row 0 = X coords, Row 1 = Y coords.
    Joints 0-16 = P0 (top-court, smaller Y), 17-33 = P1 (bottom-court).
    """
    yolo = _get_yolo_gdino()
    g_boxes, _, g_labels = _gdino_detect(bgr)
    pmask    = np.array([l == 'person' for l in g_labels], dtype=bool)
    g_player = g_boxes[pmask] if pmask.any() else g_boxes

    res = yolo(bgr, verbose=False)[0]
    out = np.zeros((2, 34), dtype=np.float32)
    if res is None or res.keypoints is None or res.boxes is None:
        return out

    y_boxes = res.boxes.xyxy.cpu().numpy()    # (N,4)
    y_cls   = res.boxes.cls.cpu().numpy()
    y_kpts  = res.keypoints.xy.cpu().numpy()  # (N,17,2)
    y_confs = res.boxes.conf.cpu().numpy()

    kept_kpts, kept_confs, kept_cy = [], [], []
    for i, (yb, yc, yconf) in enumerate(zip(y_boxes, y_cls, y_confs)):
        if int(yc) != 0:
            continue
        max_iou = max((_iou(yb, gb) for gb in g_player), default=0.0)
        if max_iou >= iou_thresh:
            kept_kpts.append(y_kpts[i])
            kept_confs.append(float(yconf))
            kept_cy.append(float(y_kpts[i][:, 1].mean()))

    if len(kept_kpts) < 2:
        # Fallback: plain YOLO top-2
        pmask2 = y_cls == 0
        if pmask2.sum() >= 2:
            order     = np.argsort(y_confs[pmask2])[::-1][:2]
            pkpts     = y_kpts[pmask2]
            kept_kpts = [pkpts[i] for i in order]
            kept_cy   = [float(kept_kpts[i][:, 1].mean()) for i in range(2)]
        else:
            return out

    if len(kept_kpts) > 2:
        order     = np.argsort(kept_confs)[::-1][:2]
        kept_kpts = [kept_kpts[i] for i in order]
        kept_cy   = [kept_cy[i]   for i in order]

    ys = np.argsort(kept_cy)
    p0, p1 = kept_kpts[ys[0]], kept_kpts[ys[1]]
    out[0, :17] = p0[:, 0];  out[1, :17] = p0[:, 1]
    out[0, 17:] = p1[:, 0];  out[1, 17:] = p1[:, 1]
    return out

print("GDINO extraction helpers ready.")


In [ ]:
# ── 7.3  GDINO extraction loop ───────────────────────────────────────────────
# All 40 rallies — already-extracted files are skipped automatically.
# On CPU: ~8-15 s/frame. On GPU (T4): ~0.3-0.5 s/frame → ~30 min total.

GDINO_RALLIES = [
    '0011_001', '0011_002', '0011_003', '0011_004', '0011_005',
    '0012_001', '0012_002', '0012_003',
    '0013_001', '0013_002', '0013_003',
    '0014_001', '0014_002', '0014_003', '0014_004', '0014_005',
    '0016_001', '0016_002', '0016_003',
    '0018_001', '0018_002', '0018_003',
    '0019_001', '0019_002', '0019_003',
    '0022_001', '0022_002', '0022_003', '0022_004', '0022_005',
    '0025_001', '0025_002', '0025_003',
    '0028_001', '0028_002', '0028_003',
    '0030_001', '0030_002', '0030_003', '0030_004',
]

GDINO_SKEL_DIR = PROJECT_ROOT / 'datasets_preprocessing' / 'finebadminton_skeletons_gdino'
GDINO_SKEL_DIR.mkdir(parents=True, exist_ok=True)
already_done = {f.stem for f in GDINO_SKEL_DIR.glob('*.npy')}
print(f"GDINO skeletons → {GDINO_SKEL_DIR}")
print(f"Already extracted: {sorted(already_done)}")
print(f"To process: {len(GDINO_RALLIES) - len(already_done)} / {len(GDINO_RALLIES)} rallies")

for rally_id in GDINO_RALLIES:
    out_path = GDINO_SKEL_DIR / f'{rally_id}.npy'
    if out_path.exists():
        print(f'[SKIP] {rally_id} — already extracted.')
        continue

    frame_list = sorted(
        [(int(fp.stem.split('_')[2]), fp)
         for fp in FB_FRAMES.glob(f'{rally_id}_*.jpg')],
        key=lambda x: x[0]
    )
    if not frame_list:
        print(f'[WARN] No frames for {rally_id}')
        continue

    T = len(frame_list)
    print(f'{rally_id}: {T} frames — starting extraction...')

    skel_arr = np.zeros((2, T, 34), dtype=np.float32)
    n_fallback = 0

    for t, (frame_num, fp) in enumerate(tqdm(frame_list, desc=rally_id)):
        bgr = cv2.imread(str(fp))
        if bgr is None:
            if t > 0:
                skel_arr[:, t, :] = skel_arr[:, t-1, :]
            continue

        sf = _extract_gdino_frame(bgr)

        if sf.sum() == 0 and t > 0:      # forward-fill on complete failure
            sf = skel_arr[:, t-1, :]
            n_fallback += 1

        skel_arr[:, t, :] = sf

    np.save(out_path, skel_arr)
    print(f'  Saved {rally_id}: {skel_arr.shape} — fallbacks: {n_fallback}/{T}')

print('Done. All files in GDINO skeleton dir:')
for f in sorted(GDINO_SKEL_DIR.glob('*.npy')):
    s = np.load(f); print(f'  {f.name}: {s.shape}')


In [ ]:

# ── 7.4  Hitter validation: Original vs GDINO ────────────────────────────────
# For each GDINO-processed rally, compare contamination rates and show:
#  - Table: per-shot Original vs GDINO P0 centroid X + contamination flag
#  - Visual grid: 4 columns (raw | original skel | GDINO skel | diff heatmap)
#    for the frames that WERE contaminated in Original

UMPIRE_X_THRESH = 200  # reuse from scan above

with open(FB_ANNOTATIONS) as _f:
    _ann_val = json.load(_f)
_rally_start_map = {r['video'].replace('.mp4',''):r['start_frame'] for r in _ann_val}
_hitting_map = {}
for _r in _ann_val:
    _rid = _r['video'].replace('.mp4','')
    _hitting_map[_rid] = (_r.get('start_frame',0), _r.get('hitting',[]))

gdino_files = sorted(GDINO_SKEL_DIR.glob('*.npy'))
if not gdino_files:
    print("No GDINO skeletons yet — run cell 7.3 first.")
else:
    for gdino_path in gdino_files:
        rid = gdino_path.stem
        orig_path = FB_SKELETONS / f'{rid}.npy'
        if not orig_path.exists():
            print(f'[SKIP] {rid} — no original skeleton to compare')
            continue

        orig_sk  = np.load(orig_path)   # (2, T, 34)
        gdino_sk = np.load(gdino_path)  # (2, T, 34)
        rs, hits = _hitting_map.get(rid, (0, []))
        T_orig  = orig_sk.shape[1]
        T_gdino = gdino_sk.shape[1]

        o_p0x = orig_sk[0,  :, :17].mean(axis=1)
        g_p0x = gdino_sk[0, :, :17].mean(axis=1)

        print(f"\n{'='*65}")
        print(f"  {rid}   orig T={T_orig}  gdino T={T_gdino}")
        print(f"{'='*65}")
        print(f"  {'Shot':<6} {'Hitter':<8} {'Strategy':<22} "
              f"{'Orig P0.x':>10} {'GDINO P0.x':>11} {'Orig?':>6} {'GDINO?':>7}")
        print(f"  {'-'*75}")

        orig_cont = 0; gdino_cont = 0; n_shots = 0
        for i, hit in enumerate(hits):
            strats = hit.get('strategies') or []
            strat  = strats[0] if strats else '?'
            hitter = hit.get('hitter','?')
            hf     = hit.get('hit_frame', 0)
            rel    = hf - rs
            s      = max(0, rel - 8)

            o_end = min(T_orig, s + 16)
            g_end = min(T_gdino, s + 16)

            o_win = o_p0x[s:o_end]
            g_win = g_p0x[s:g_end]

            o_bad = int((o_win < UMPIRE_X_THRESH).sum())
            g_bad = int((g_win < UMPIRE_X_THRESH).sum())

            o_mean = float(o_win.mean()) if len(o_win) else 0.0
            g_mean = float(g_win.mean()) if len(g_win) else 0.0

            o_flag = f"BAD({o_bad})" if o_bad else "ok"
            g_flag = f"BAD({g_bad})" if g_bad else "ok"

            n_shots += 1
            if o_bad: orig_cont  += 1
            if g_bad: gdino_cont += 1

            print(f"  {i:<6} {hitter:<8} {strat[:20]:<22} "
                  f"{o_mean:>10.1f} {g_mean:>11.1f} {o_flag:>6} {g_flag:>7}")

        print(f"\n  Contaminated shots  —  Original: {orig_cont}/{n_shots} "
              f"({100*orig_cont/max(n_shots,1):.0f}%)"
              f"  |  GDINO: {gdino_cont}/{n_shots} "
              f"({100*gdino_cont/max(n_shots,1):.0f}%)")

        # ── Visual grid: contaminated frames comparison ────────────────────
        bad_frames = [t for t in range(T_orig) if o_p0x[t] < UMPIRE_X_THRESH][:6]
        if not bad_frames:
            print("  No contaminated frames found in original — nothing to show.")
            continue

        fig, axes = plt.subplots(len(bad_frames), 3,
                                 figsize=(21, 5.5 * len(bad_frames)))
        if len(bad_frames) == 1:
            axes = axes[np.newaxis, :]

        frame_list_rid = sorted(
            [(int(fp.stem.split('_')[2]), fp)
             for fp in FB_FRAMES.glob(f'{rid}_*.jpg')],
            key=lambda x: x[0]
        )

        for row, t in enumerate(bad_frames):
            if t >= len(frame_list_rid):
                continue
            abs_fn, fp = frame_list_rid[t]
            bgr = cv2.imread(str(fp))
            if bgr is None:
                continue

            orig_sf  = orig_sk[:, t, :]                             # (2,34)
            gdino_sf = gdino_sk[:, min(t, T_gdino-1), :]           # (2,34)

            def _draw_skel(bgr_in, sf, label):
                img = cv2.cvtColor(bgr_in, cv2.COLOR_BGR2RGB).copy()
                ov  = img.copy()
                for p, col in enumerate([(0,200,255),(255,120,0)]):
                    b = p * 17
                    xs, ys = sf[0,b:b+17], sf[1,b:b+17]
                    for j1,j2 in [(0,1),(0,2),(1,3),(2,4),(5,6),(5,7),(7,9),
                                  (6,8),(8,10),(5,11),(6,12),(11,12),(11,13),
                                  (13,15),(12,14),(14,16)]:
                        x1,y1,x2,y2 = int(xs[j1]),int(ys[j1]),int(xs[j2]),int(ys[j2])
                        if x1>0 and y1>0 and x2>0 and y2>0:
                            cv2.line(ov,(x1,y1),(x2,y2),col,2)
                    for j in range(17):
                        x,y = int(xs[j]),int(ys[j])
                        if x>0 and y>0:
                            cv2.circle(ov,(x,y),5,col,-1)
                    ok = (xs>0)&(ys>0)
                    if ok.any():
                        cx,cy = int(xs[ok].mean()),int(ys[ok].mean())
                        cv2.putText(ov,f'P{p}({cx},{cy})',(cx-20,cy-12),
                                    cv2.FONT_HERSHEY_SIMPLEX,0.5,col,2)
                return cv2.addWeighted(ov,0.85,img,0.15,0)

            axes[row,0].imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
            axes[row,0].set_title(f'{rid} t={t} abs={abs_fn}\nRaw frame', fontsize=8)
            axes[row,0].axis('off')

            axes[row,1].imshow(_draw_skel(bgr, orig_sf, 'Original'))
            p0x_o = orig_sf[0,:17].mean()
            axes[row,1].set_title(
                f'Original YOLOv8\nP0.x={p0x_o:.0f}  '
                f'{"⚠ UMPIRE" if p0x_o < UMPIRE_X_THRESH else "ok"}', fontsize=8)
            axes[row,1].axis('off')

            axes[row,2].imshow(_draw_skel(bgr, gdino_sf, 'GDINO'))
            p0x_g = gdino_sf[0,:17].mean()
            axes[row,2].set_title(
                f'GDINO-guided\nP0.x={p0x_g:.0f}  '
                f'{"⚠ UMPIRE" if p0x_g < UMPIRE_X_THRESH else "ok"}', fontsize=8)
            axes[row,2].axis('off')

        plt.suptitle(
            f'{rid}  —  Contaminated frames: Original vs GDINO-guided\n'
            'Cyan=P0 (top-court)  Orange=P1 (bottom-court)  ⚠=chair umpire detected as P0',
            fontsize=10
        )
        plt.tight_layout()
        plt.show()
